# 06 — AI Development Lifecycle & MLOps Infrastructure — Lab

This lab instruments a scikit-learn logistic-regression training script with MLflow. You will create three tracked runs on the Iris dataset, compare validation accuracy, and register the best model artifact. The exercise covers model versioning and experiment tracking with MLflow from the lecture, while DVC, CI/CD, and deployment patterns remain lecture concepts.

## Objectives
This lab exercises a subset of the lesson objectives:
- Instrument a training script with MLflow to log parameters, metrics, and artifacts across multiple runs, and compare those runs in the MLflow UI. *(hands-on in this lab)*
- Register the best model artifact as a named model version. *(hands-on in this lab)*

The lecture also covers lifecycle stages, DVC versioning, and deployment patterns. This 10-minute lab focuses on the MLflow operations only.

## Prerequisites
- Read the lecture sections on ML pipeline stages, MLflow, and model registration.
- Know Python functions, dictionaries, file paths, and basic scikit-learn model evaluation.
- Know model training and held-out validation from m05.

## Required Software and Packages
- Python 3.10 or newer
- mlflow 2.10 or newer
- scikit-learn 1.3 or newer
- joblib 1.3 or newer

## Environment and Setup
Install the listed packages in the active environment before running the exercise. The code uses a local SQLite tracking store, so no external tracking server is required.

In [ ]:
import json
import os
import tempfile
from pathlib import Path

import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

## Background
MLflow assigns each training execution a run record. Parameters describe configuration, metrics describe measured results, and artifacts store files such as a model or metrics report. A registered model gives the best artifact a stable name and version for later promotion.

## Exercise Instructions
1. Complete the training function so one MLflow run logs the `C` parameter, validation accuracy, metrics JSON, and model artifact. Start the local MLflow UI with `mlflow ui --backend-store-uri sqlite:////tmp/m06-mlflow.db` in a terminal when ready. (2 min)
2. Run the completed function with `C=0.01`, `C=1.0`, and `C=100.0`. Open the MLflow UI and confirm three runs show the parameter and validation accuracy. (5 min)
3. Register the best run's model URI under the name `iris-logreg`, then confirm model version 1 appears in the Model Registry. (3 min)

In [ ]:
TRACKING_URI = "sqlite:////tmp/m06-mlflow.db"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("m06-iris-logreg")

def train_and_track(c_value):
    iris = load_iris()
    features_train, features_valid, labels_train, labels_valid = train_test_split(
        iris.data, iris.target, test_size=0.25, random_state=42, stratify=iris.target
    )
    # TODO: open an MLflow run and log the parameter, metric, JSON artifact, and model.
    return None

In [ ]:
def train_and_track(c_value):
    iris = load_iris()
    features_train, features_valid, labels_train, labels_valid = train_test_split(
        iris.data, iris.target, test_size=0.25, random_state=42, stratify=iris.target
    )
    model = LogisticRegression(C=c_value, max_iter=500, random_state=42)
    with mlflow.start_run() as run:
        model.fit(features_train, labels_train)
        predictions = model.predict(features_valid)
        validation_accuracy = accuracy_score(labels_valid, predictions)
        metrics_path = Path(tempfile.gettempdir()) / f"m06-metrics-{run.info.run_id}.json"
        metrics_path.write_text(json.dumps({"validation_accuracy": validation_accuracy}))
        mlflow.log_param("C", c_value)
        mlflow.log_metric("validation_accuracy", validation_accuracy)
        mlflow.log_artifact(str(metrics_path))
        mlflow.sklearn.log_model(model, "model")
        return run.info.run_id, validation_accuracy

In [ ]:
run_results = [train_and_track(c_value) for c_value in (0.01, 1.0, 100.0)]
best_run_id, best_accuracy = max(run_results, key=lambda result: result[1])
print(f"runs={len(run_results)} best_accuracy={best_accuracy:.3f}")

## Expected Outputs
The previous cell prints `runs=3` and a validation accuracy between `0.90` and `1.00`. The MLflow UI lists three runs with `C` values `0.01`, `1.0`, and `100.0`, plus a `validation_accuracy` metric and a model artifact for each run.

## Questions and Tasks
1. Which logged field identifies the regularization setting for each run?
2. Why should the registered model point to the best run's artifact rather than a local filename?

In [ ]:
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("m06-iris-logreg")
runs = client.search_runs(experiment_ids=[experiment.experiment_id])
run_cs = {run.data.params["C"] for run in runs}
assert len(runs) >= 3
assert {"0.01", "1.0", "100.0"}.issubset(run_cs)
assert all("validation_accuracy" in run.data.metrics for run in runs)
best_run = max(runs, key=lambda run: run.data.metrics["validation_accuracy"])
model_uri = f"runs:/{best_run.info.run_id}/model"
registered = mlflow.register_model(model_uri, "iris-logreg")
assert registered.version == 1
print("PASS: three tracked runs and iris-logreg version 1")

## Optional
Add a descriptive run tag for the dataset version, then filter the MLflow UI by the tag.

## Completion Criteria
- Three MLflow runs appear with `C` values `0.01`, `1.0`, and `100.0`.
- Each run contains a `validation_accuracy` metric and a model artifact.
- The verification cell prints `PASS: three tracked runs and iris-logreg version 1`.
- Model Registry contains `iris-logreg` version `1`.